In [ ]:

# ============================================================
# USE CASE 3: CUSTOMER FEEDBACK INTELLIGENCE
# TF-IDF + K-MEANS + SENTIMENT ANALYSIS + DATA ENRICHMENT
# Snowflake Warehouse Runtime | Single Notebook Cell
# ============================================================

import re
import json
import warnings
from datetime import datetime, timezone
from collections import Counter

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

from snowflake.snowpark.context import get_active_session

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

DATABASE_NAME = "API_DATA_ENGINEERING"
RAW_SCHEMA = "AI_RAW"
ENRICHED_SCHEMA = "AI_ENRICHED"
ANALYTICS_SCHEMA = "AI_ANALYTICS"

SOURCE_TABLE = f"{DATABASE_NAME}.{RAW_SCHEMA}.CUSTOMER_FEEDBACK"
TARGET_TABLE = f"{DATABASE_NAME}.{ENRICHED_SCHEMA}.CUSTOMER_FEEDBACK_AI"
TEMP_TABLE = f"{DATABASE_NAME}.{ENRICHED_SCHEMA}.TEMP_CUSTOMER_FEEDBACK_ML"

MODEL_NAME = "TFIDF_KMEANS_CUSTOMER_INTELLIGENCE"
NUMBER_OF_TOPICS = 5

# ------------------------------------------------------------
# 2. GET SNOWFLAKE SESSION AND SET CONTEXT
# ------------------------------------------------------------

session = get_active_session()

session.sql(f"USE DATABASE {DATABASE_NAME}").collect()
session.sql(f"USE SCHEMA {ENRICHED_SCHEMA}").collect()

print("Snowflake session initialized.")
print("Database:", DATABASE_NAME)
print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)

# ------------------------------------------------------------
# 3. LOAD UNPROCESSED CUSTOMER FEEDBACK
# ------------------------------------------------------------

feedback_df = (
    session.table(SOURCE_TABLE)
    .filter("IS_PROCESSED = FALSE")
)

source_rows = feedback_df.collect()

# Remove duplicate FEEDBACK_ID values only.
# Different feedback IDs are retained even if their text is identical.

records_by_id = {}

for row in source_rows:
    record = row.as_dict()
    feedback_id = record.get("FEEDBACK_ID")

    if feedback_id is not None and feedback_id not in records_by_id:
        records_by_id[feedback_id] = record

records = list(records_by_id.values())

duplicate_id_count = len(source_rows) - len(records)

print("\nSOURCE DATA")
print("Source rows:", len(source_rows))
print("Unique feedback IDs:", len(records))
print("Duplicate IDs skipped:", duplicate_id_count)

if not records:
    print("\nNo unprocessed feedback records found.")
else:

    # --------------------------------------------------------
    # 4. PREPARE FEEDBACK TEXT
    # --------------------------------------------------------

    feedback_texts = [
        str(record.get("FEEDBACK_TEXT") or "").strip()
        for record in records
    ]

    # Avoid empty documents in the ML model
    feedback_texts = [
        text if text else "no feedback provided"
        for text in feedback_texts
    ]

    print("\nFeedback records loaded:", len(feedback_texts))

    # --------------------------------------------------------
    # 5. TF-IDF TEXT VECTORIZATION
    # --------------------------------------------------------

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=1000,
        min_df=1,
        sublinear_tf=True
    )

    tfidf_matrix = vectorizer.fit_transform(feedback_texts)

    feature_names = vectorizer.get_feature_names_out()

    print("TF-IDF feature matrix:", tfidf_matrix.shape)

    # --------------------------------------------------------
    # 6. DISCOVER TOPICS USING K-MEANS
    # --------------------------------------------------------

    # Avoid creating unnecessary clusters when documents
    # contain identical or zero-valued feature vectors.

    unique_vectors = np.unique(tfidf_matrix.toarray(), axis=0)
    number_of_topics = min(NUMBER_OF_TOPICS, len(unique_vectors))

    number_of_topics = max(1, number_of_topics)

    kmeans = KMeans(
        n_clusters=number_of_topics,
        random_state=42,
        n_init=10,
        max_iter=300
    )

    cluster_ids = kmeans.fit_predict(tfidf_matrix)

    # Extract representative keywords for each topic

    topic_keywords = {}

    for topic_id in range(number_of_topics):

        center = kmeans.cluster_centers_[topic_id]

        top_indices = center.argsort()[-5:][::-1]

        keywords = [
            feature_names[index]
            for index in top_indices
            if center[index] > 0
        ]

        if not keywords:
            keywords = ["general feedback"]

        topic_keywords[topic_id] = keywords

    topic_labels = {
        topic_id: " / ".join(
            word.upper() for word in keywords[:3]
        )
        for topic_id, keywords in topic_keywords.items()
    }

    print("\nDISCOVERED CUSTOMER FEEDBACK TOPICS")

    for topic_id, keywords in topic_keywords.items():
        print(
            f"Topic {topic_id}:",
            ", ".join(keywords)
        )

    # --------------------------------------------------------
    # 7. SENTIMENT ANALYSIS
    # --------------------------------------------------------

    positive_words = {
        "excellent", "good", "great", "happy", "satisfied",
        "fast", "helpful", "perfect", "amazing", "love",
        "recommend", "reliable", "smooth", "friendly",
        "quick", "easy", "quality", "wonderful", "impressed"
    }

    negative_words = {
        "bad", "poor", "late", "slow", "broken", "damaged",
        "disappointed", "terrible", "problem", "issue",
        "rude", "expensive", "difficult", "defective",
        "unhappy", "worst", "delay", "unreliable",
        "failed", "missing", "complaint"
    }

    negation_words = {
        "not", "no", "never", "neither", "hardly",
        "without", "cannot", "can't", "didn't", "doesn't",
        "wasn't", "isn't"
    }

    intensifier_words = {
        "very", "extremely", "really", "highly", "absolutely"
    }

    def analyse_sentiment(text):

        tokens = re.findall(r"[a-z']+", text.lower())

        score = 0.0

        for index, word in enumerate(tokens):

            if word in positive_words:
                word_score = 1.0
            elif word in negative_words:
                word_score = -1.0
            else:
                continue

            # Apply nearby negation
            previous_words = tokens[max(0, index - 3):index]

            if any(w in negation_words for w in previous_words):
                word_score *= -1

            # Apply intensifier
            if any(w in intensifier_words for w in previous_words):
                word_score *= 1.5

            score += word_score

        # Normalize score to the range -1 to +1
        normalized_score = max(
            -1.0,
            min(1.0, score / max(1.0, len(tokens) * 0.12))
        )

        if normalized_score > 0.10:
            sentiment = "POSITIVE"
        elif normalized_score < -0.10:
            sentiment = "NEGATIVE"
        else:
            sentiment = "NEUTRAL"

        return sentiment, round(normalized_score, 3)

    # --------------------------------------------------------
    # 8. RECOMMENDED ACTION
    # --------------------------------------------------------

    def recommend_action(sentiment, text):

        text_lower = text.lower()

        urgent_terms = [
            "broken", "damaged", "defective",
            "failed", "worst", "rude"
        ]

        if sentiment == "NEGATIVE":

            if any(term in text_lower for term in urgent_terms):
                return "PRIORITIZE_CUSTOMER_COMPLAINT"

            return "INVESTIGATE_CUSTOMER_COMPLAINT"

        elif sentiment == "NEUTRAL":
            return "MONITOR_CUSTOMER_FEEDBACK"

        return "MAINTAIN_CURRENT_SERVICE"

    # --------------------------------------------------------
    # 9. STRUCTURED INFORMATION EXTRACTION
    # --------------------------------------------------------

    def extract_information(text, topic_id):

        words = re.findall(r"[a-z']+", text.lower())

        keywords = [
            word for word, count in Counter(words).most_common(10)
            if len(word) > 3
        ][:5]

        return {
            "cluster_id": int(topic_id),
            "discovered_topic": topic_labels[topic_id],
            "important_keywords": keywords,
            "topic_keywords": topic_keywords[topic_id]
        }

    # --------------------------------------------------------
    # 10. BUILD ENRICHED OUTPUT RECORDS
    # --------------------------------------------------------

    output_rows = []

    processed_at = datetime.now(timezone.utc).replace(tzinfo=None)

    for index, record in enumerate(records):

        feedback_text = feedback_texts[index]

        topic_id = int(cluster_ids[index])

        sentiment, sentiment_score = analyse_sentiment(
            feedback_text
        )

        action = recommend_action(
            sentiment,
            feedback_text
        )

        extracted_information = extract_information(
            feedback_text,
            topic_id
        )

        extracted_information["recommended_action"] = action

        topic_label = topic_labels[topic_id]

        ai_summary = (
            f"{sentiment} customer feedback about "
            f"{topic_label}. "
            f"Recommended action: {action}."
        )

        output_rows.append((
            record.get("FEEDBACK_ID"),
            record.get("CUSTOMER_ID"),
            record.get("CUSTOMER_NAME"),
            record.get("PRODUCT_NAME"),
            record.get("FEEDBACK_TEXT"),
            sentiment,
            float(sentiment_score),
            topic_label,
            json.dumps(extracted_information),
            ai_summary,
            processed_at,
            MODEL_NAME
        ))

    print("\nENRICHMENT COMPLETED")
    print("Records enriched:", len(output_rows))

    # --------------------------------------------------------
    # 11. CREATE SNOWPARK DATAFRAME
    # --------------------------------------------------------

    output_schema = [
        "FEEDBACK_ID",
        "CUSTOMER_ID",
        "CUSTOMER_NAME",
        "PRODUCT_NAME",
        "FEEDBACK_TEXT",
        "SENTIMENT",
        "SENTIMENT_SCORE",
        "FEEDBACK_CATEGORY",
        "EXTRACTED_INFORMATION_JSON",
        "AI_SUMMARY",
        "AI_PROCESSED_AT",
        "MODEL_NAME"
    ]

    output_df = session.create_dataframe(
        output_rows,
        schema=output_schema
    )

    # --------------------------------------------------------
    # 12. WRITE TO FULLY QUALIFIED TEMPORARY STAGING TABLE
    # --------------------------------------------------------

    output_df.write.mode("overwrite").save_as_table(
        TEMP_TABLE,
        table_type="temporary"
    )

    staging_count = session.table(TEMP_TABLE).count()

    print("\nSTAGING TABLE CREATED")
    print("Staging records:", staging_count)
    print("Staging table:", TEMP_TABLE)

    # --------------------------------------------------------
    # 13. MERGE ENRICHED DATA INTO TARGET TABLE
    # --------------------------------------------------------

    merge_sql = f"""
        MERGE INTO {TARGET_TABLE} AS T
        USING {TEMP_TABLE} AS S
            ON T.FEEDBACK_ID = S.FEEDBACK_ID

        WHEN MATCHED THEN UPDATE SET
            T.CUSTOMER_ID = S.CUSTOMER_ID,
            T.CUSTOMER_NAME = S.CUSTOMER_NAME,
            T.PRODUCT_NAME = S.PRODUCT_NAME,
            T.FEEDBACK_TEXT = S.FEEDBACK_TEXT,
            T.SENTIMENT = S.SENTIMENT,
            T.SENTIMENT_SCORE = S.SENTIMENT_SCORE,
            T.FEEDBACK_CATEGORY = S.FEEDBACK_CATEGORY,
            T.EXTRACTED_INFORMATION =
                PARSE_JSON(S.EXTRACTED_INFORMATION_JSON),
            T.AI_SUMMARY = S.AI_SUMMARY,
            T.AI_PROCESSED_AT = S.AI_PROCESSED_AT,
            T.MODEL_NAME = S.MODEL_NAME

        WHEN NOT MATCHED THEN INSERT (
            FEEDBACK_ID,
            CUSTOMER_ID,
            CUSTOMER_NAME,
            PRODUCT_NAME,
            FEEDBACK_TEXT,
            SENTIMENT,
            SENTIMENT_SCORE,
            FEEDBACK_CATEGORY,
            EXTRACTED_INFORMATION,
            AI_SUMMARY,
            AI_PROCESSED_AT,
            MODEL_NAME
        )
        VALUES (
            S.FEEDBACK_ID,
            S.CUSTOMER_ID,
            S.CUSTOMER_NAME,
            S.PRODUCT_NAME,
            S.FEEDBACK_TEXT,
            S.SENTIMENT,
            S.SENTIMENT_SCORE,
            S.FEEDBACK_CATEGORY,
            PARSE_JSON(S.EXTRACTED_INFORMATION_JSON),
            S.AI_SUMMARY,
            S.AI_PROCESSED_AT,
            S.MODEL_NAME
        )
    """

    session.sql(merge_sql).collect()

    print("\nTARGET TABLE MERGE COMPLETED")

    # --------------------------------------------------------
    # 14. MARK SOURCE RECORDS AS PROCESSED
    # --------------------------------------------------------

    update_sql = f"""
        UPDATE {SOURCE_TABLE} AS R
        SET IS_PROCESSED = TRUE
        WHERE EXISTS (
            SELECT 1
            FROM {TEMP_TABLE} AS S
            WHERE R.FEEDBACK_ID = S.FEEDBACK_ID
        )
    """

    session.sql(update_sql).collect()

    print("Source records marked as processed.")

    # --------------------------------------------------------
    # 15. DISPLAY ENRICHED CUSTOMER FEEDBACK
    # --------------------------------------------------------

    print("\nSAMPLE ENRICHED CUSTOMER FEEDBACK")

    session.table(TARGET_TABLE) \
        .filter(f"MODEL_NAME = '{MODEL_NAME}'") \
        .sort("FEEDBACK_ID") \
        .show(8)

    # --------------------------------------------------------
    # 16. TOPIC AND SENTIMENT ANALYTICS
    # --------------------------------------------------------

    print("\nCUSTOMER FEEDBACK TOPIC SUMMARY")

    summary_sql = f"""
        SELECT
            FEEDBACK_CATEGORY,
            SENTIMENT,
            COUNT(*) AS FEEDBACK_COUNT,
            ROUND(AVG(SENTIMENT_SCORE), 3) AS AVG_SENTIMENT_SCORE
        FROM {TARGET_TABLE}
        WHERE MODEL_NAME = '{MODEL_NAME}'
        GROUP BY
            FEEDBACK_CATEGORY,
            SENTIMENT
        ORDER BY
            FEEDBACK_COUNT DESC
    """

    session.sql(summary_sql).show()

    # --------------------------------------------------------
    # 17. FINAL VALIDATION
    # --------------------------------------------------------

    final_count = (
        session.table(TARGET_TABLE)
        .filter(f"MODEL_NAME = '{MODEL_NAME}'")
        .count()
    )

    remaining_count = (
        session.table(SOURCE_TABLE)
        .filter("IS_PROCESSED = FALSE")
        .count()
    )

    print("\n======================================")
    print("CUSTOMER FEEDBACK ML PIPELINE FINISHED")
    print("======================================")
    print("Unique source records selected:", len(records))
    print("Staging records:", staging_count)
    print("Enriched target records:", final_count)
    print("Remaining unprocessed source records:", remaining_count)
    print("Model:", MODEL_NAME)
    print("======================================")
